# Ch 09 · Contacts, Constraint Budgets and Overflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/09-contacts-and-buffer-sizing.ipynb)

Companion notebook for the chapter page `chapters/09-contacts-and-buffer-sizing.html` ([page](../chapters/09-contacts-and-buffer-sizing.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## What this notebook covers
Every code cell below is the same file as on the chapter page (`examples/09-contacts-and-buffer-sizing/`). On a GPU the budgets grow automatically (`NWORLD = 4096 if ON_GPU else 8`). Numbers you see on a GPU differ from the CPU outputs on the page in contact order, which world starves, and timings.

### The running project
`my_env.py` after Chapter 9: Ch 6 `make_worlds`, Ch 8 resets and snapshots, and this chapter's `describe_overflow(d)` (Example 9.7).

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 9.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 adds describe_overflow (Example 9.7).
Later chapters extend this file (Ch 10 randomize, Ch 14 sensor_slices,
Ch 24 capture_step, Ch 26 obs/reward kernels, Ch 30 the environments).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done, randomize=None):
    """Reset the worlds whose done flag is True, randomize them, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). `randomize(m, d, done)`, if given, launches
    the per-episode randomization (initial velocity, targets, ...) of the done worlds. It runs
    after reset_data and before forward, so every derived quantity (xpos, cvel, velocity sensors,
    qfrc_bias, ...) matches the final state. Everything stays on the device, so this can be
    captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    if randomize is not None:
        randomize(m, d, done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS (+ State.TIME if anything reads d.time) for MPC (it includes HISTORY and
    PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)

## 2. The Contact struct and the shared pool
`d.contact` has one flat buffer of `naconmax` slots shared by all worlds; each contact stores its `worldid`. `d.nacon` (shape `(1,)`) is a global counter that can exceed the capacity: always read `n = min(d.nacon.numpy()[0], d.naconmax)`.

In [ ]:
# Example 9.1 - ex1_read_contacts.py
"""Example 9.1 - read the shared contact pool: one table row per contact, over 4 worlds.

A sphere, a box and a capsule rest on a plane. World 2 lifts the box 0.5 m, so it has fewer
contacts. We run mjw.forward once and print every field of d.contact that matters.
Run: python ex1_read_contacts.py [--dump-json PATH]
  --dump-json PATH writes the first contact's fields for the page's contact card.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import ContactType   # private API: not exported by mujoco_warp

np.set_printoptions(precision=4, suppress=True)

CONTACT_SCENE = """
<mujoco model="contact_scene">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="sphere" pos="0 0 0.09"><freejoint/>
      <geom name="sphere" type="sphere" size="0.1"/></body>
    <body name="box" pos="1 0 0.095"><freejoint/>
      <geom name="box" type="box" size="0.1 0.1 0.1"/></body>
    <body name="capsule" pos="2 0 0.09" euler="0 90 0"><freejoint/>
      <geom name="capsule" type="capsule" size="0.1 0.2"/></body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 4          # the table is meant to be read: 4 worlds on CPU and GPU

mjm = mujoco.MjModel.from_xml_string(CONTACT_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[2, 9] += 0.5                       # world 2: box body (qpos 7..13) lifted 0.5 m
d.qpos.assign(qpos)
mjw.forward(m, d)                       # kinematics -> collision -> make_constraint -> solve

nacon = int(d.nacon.numpy()[0])
n = min(nacon, d.naconmax)              # the counter can exceed capacity; never read past it
print(f"naconmax = {d.naconmax} (nconmax 16 x nworld {NWORLD}) | nacon = {nacon}"
      f" | ncollision = {int(d.ncollision.numpy()[0])}")

c = d.contact
world, geom = c.worldid.numpy()[:n], c.geom.numpy()[:n]
dist, pos, frame = c.dist.numpy()[:n], c.pos.numpy()[:n], c.frame.numpy()[:n]
dim, ctype, efc = c.dim.numpy()[:n], c.type.numpy()[:n], c.efc_address.numpy()[:n]
gname = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]

print(" cid  w  geom[0]-geom[1]     dist     pos (midpoint)          normal = frame[0]  dim"
      "  type        efc_address")
for i in range(n):
    pair = f"{gname[geom[i, 0]]}-{gname[geom[i, 1]]}"
    p = np.array2string(pos[i], precision=3, suppress_small=True)
    nrm = np.array2string(frame[i, 0], precision=1, suppress_small=True)
    print(f"{i:4d} {world[i]:2d}  {pair:16s} {dist[i]:8.4f}  {p:22s}  {nrm:17s}  {dim[i]:3d}"
          f"  {ContactType(int(ctype[i])).name:10s}  {efc[i]}")

print("contacts per world:", np.bincount(world, minlength=NWORLD))
print("nefc per world:    ", d.nefc.numpy(), "(4 pyramidal rows per condim-3 contact)")

# One contact in full: the fields that do not fit in the table.
k = 0
print("\ncontact 0 in full")
print("  frame rows (normal, tangent1, tangent2):\n", frame[k])
print("  friction (t1, t2, spin, roll1, roll2):", c.friction.numpy()[k])
print("  solref:", c.solref.numpy()[k], " solreffriction:", c.solreffriction.numpy()[k])
print("  solimp:", c.solimp.numpy()[k])
print("  includemargin:", c.includemargin.numpy()[k], " adhesion:", c.adhesion.numpy()[k],
      " geomcollisionid:", c.geomcollisionid.numpy()[k])
print("  surfacevel:", c.surfacevel.numpy()[k])
print("  flex/elem/vert shapes (nflex = 0):", c.flex.shape, c.elem.shape, c.vert.shape)
print("  efc_address shape (naconmax, nmaxpyramid):", c.efc_address.shape)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    def r4(a):                          # float64 before rounding: clean JSON numbers
        return np.round(np.asarray(a, dtype=float), 4).tolist()

    card = {"world": int(world[k]), "geom": [gname[g] for g in geom[k]],
            "dist": round(float(dist[k]), 4),
            "pos": r4(pos[k]), "frame": r4(frame[k]),
            "friction": r4(c.friction.numpy()[k]), "dim": int(dim[k]),
            "type": int(ctype[k]), "solref": r4(c.solref.numpy()[k]),
            "solreffriction": r4(c.solreffriction.numpy()[k]),
            "solimp": r4(c.solimp.numpy()[k]),
            "includemargin": float(c.includemargin.numpy()[k]),
            "surfacevel": r4(c.surfacevel.numpy()[k]),
            "efc_address": efc[k].tolist(), "nefc_world0": int(d.nefc.numpy()[0]),
            "rows": [{"cid": int(i), "w": int(world[i]), "pair": [gname[g] for g in geom[i]],
                      "dist": round(float(dist[i]), 4), "efc": efc[i].tolist()} for i in range(n)]}
    with open(args.dump_json, "w") as f:
        json.dump(card, f, separators=(",", ":"))
    print("wrote", args.dump_json)

Group the pool per world with `np.bincount` (counts) and `np.minimum.at` (deepest penetration).

In [ ]:
# Example 9.2 - ex2_per_world.py
"""Example 9.2 - group the flat contact pool by world: counts and deepest penetration per world.

Same scene as Example 9.1. Each world lifts a random subset of the three objects, so the
worlds have different numbers of contacts. np.bincount and np.minimum.at do the grouping.
Run: python ex2_per_world.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

CONTACT_SCENE = """
<mujoco model="contact_scene">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="sphere" pos="0 0 0.09"><freejoint/>
      <geom name="sphere" type="sphere" size="0.1"/></body>
    <body name="box" pos="1 0 0.095"><freejoint/>
      <geom name="box" type="box" size="0.1 0.1 0.1"/></body>
    <body name="capsule" pos="2 0 0.09" euler="0 90 0"><freejoint/>
      <geom name="capsule" type="capsule" size="0.1 0.2"/></body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(CONTACT_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

rng = np.random.default_rng(1)
lift = rng.integers(0, 2, size=(NWORLD, 3)).astype(bool)    # which objects are lifted, per world
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
for k, z_index in enumerate((2, 9, 16)):                   # z of sphere, box, capsule free joints
    qpos[lift[:, k], z_index] += 0.5
qpos[:, 2] -= rng.uniform(0.0, 0.01, NWORLD)               # sphere sinks by a random extra amount
d.qpos.assign(qpos)
mjw.forward(m, d)

n = min(int(d.nacon.numpy()[0]), d.naconmax)
world = d.contact.worldid.numpy()[:n]
dist = d.contact.dist.numpy()[:n]

per_world = np.bincount(world, minlength=d.nworld)          # contacts per world
deepest = np.full(d.nworld, np.inf)                          # +inf = no contact in that world
np.minimum.at(deepest, world, dist)                          # unbuffered min, handles repeats

print(f"nacon = {n} contacts in one pool of naconmax = {d.naconmax}")
print("world  lifted(s,b,c)  contacts  deepest dist  nefc")
nefc = d.nefc.numpy()
for w in range(min(NWORLD, 8)):
    deep = f"{deepest[w]:.4f}" if np.isfinite(deepest[w]) else "  none"
    print(f"{w:5d}  {lift[w].astype(int)}       {per_world[w]:8d}  {deep:>12s}  {nefc[w]:4d}")
print("check: sum of per-world counts == nacon:", per_world.sum() == n)
print("max contacts in one world:", per_world.max(), "| nconmax was 16")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 3. Which way the normal points
`geom[0]` has the lower GeomType; `frame[0]` points from `geom[0]` to `geom[1]`.

In [ ]:
# Example 9.3 - ex3_normal_direction.py
"""Example 9.3 - which geom is geom[0], and which way the normal points.

The XML lists the ball BEFORE the plane, and a box (id 2) BEFORE the pebble (id 3) that rests
on it. Neither XML order nor geom id decides: MJWarp stores the lower GeomType first
(PLANE=0 < SPHERE=2 < BOX=6), and frame[0] (the normal) points from geom[0] to geom[1].
Run: python ex3_normal_direction.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

ORDER_SCENE = """
<mujoco model="normal_order">
  <worldbody>
    <body name="ball" pos="0 0 0.099"><freejoint/>
      <geom name="ball" type="sphere" size="0.1"/></body>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="crate" pos="1 0 0.099"><freejoint/>
      <geom name="crate" type="box" size="0.1 0.1 0.1"/></body>
    <body name="pebble" pos="1 0 0.2485"><freejoint/>
      <geom name="pebble" type="sphere" size="0.05"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(ORDER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
mjw.forward(m, d)

names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
print("geom ids after compiling (world-body geoms first):", names)
print("GeomType per geom: ", [mjw.GeomType(int(t)).name for t in mjm.geom_type])

n = min(int(d.nacon.numpy()[0]), d.naconmax)
geom = d.contact.geom.numpy()[:n]
normal = d.contact.frame.numpy()[:n, 0]
dist = d.contact.dist.numpy()[:n]
seen = set()
for i in range(n):
    g0, g1 = geom[i]
    if (g0, g1) in seen:            # the box-plane pair has 4 contacts; print one per pair
        continue
    seen.add((g0, g1))
    print(f"geom[0] = {names[g0]:6s} ({mjw.GeomType(int(mjm.geom_type[g0])).name:6s})  "
          f"geom[1] = {names[g1]:6s} ({mjw.GeomType(int(mjm.geom_type[g1])).name:6s})  "
          f"normal = {normal[i]}  dist = {dist[i]:.4f}")
print("rule: normal points from geom[0] to geom[1]; push geom[1] along +normal to separate")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 4. Friction cones, rows per contact and contact forces
A condim-c contact costs `2(c-1)` rows with the default pyramidal cone (`c` with elliptic). `mjw.contact_force` turns a contact's rows back into a 6-D force:torque.

In [ ]:
# Example 9.4 - ex4_contact_force.py
"""Example 9.4 - contact forces with mjw.contact_force, checked against Newton's law and C MuJoCo.

PLAYGROUND settles for 2 s. mjw.contact_force turns each contact's efc forces into a 6-D
force:torque, here in the world frame. The floor must hold up the 1 kg ball with ~9.81 N and the
8 kg box with ~78.5 N. get_data_into copies one world (and only its contacts) into an MjData.
Run: python ex4_contact_force.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200                                   # 2 s at dt = 0.01: everything has settled

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(NSTEP):
    mjw.step(m, d)

# one force:torque per contact slot; slots past nacon stay zero (the kernel returns early)
ids = wp.array(np.arange(d.naconmax), dtype=int)
force = wp.zeros(d.naconmax, dtype=wp.spatial_vector)
mjw.contact_force(m, d, ids, True, force)    # to_world_frame=True

n = min(int(d.nacon.numpy()[0]), d.naconmax)
f = force.numpy()[:n]                         # (n, 6): force xyz, then torque xyz
world = d.contact.worldid.numpy()[:n]
geom1 = d.contact.geom.numpy()[:n, 1]         # geom[0] = floor; geom[1] is pushed along +normal
ball_geom = mjm.body_geomadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "ball")]
box_geom = mjm.body_geomadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")]

g = -mjm.opt.gravity[2]
for name, gid, body in (("ball", ball_geom, "ball"), ("box", box_geom, "box")):
    mass = mjm.body_mass[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, body)]
    sel = (geom1 == gid) & (world == 0)
    fz = f[sel, 2].sum()
    print(f"world 0 {name}: {sel.sum()} contact(s), total normal force {fz:7.3f} N"
          f"   m*g = {mass * g:7.3f} N")

per_world_ball = np.zeros(NWORLD)
np.add.at(per_world_ball, world[geom1 == ball_geom], f[geom1 == ball_geom, 2])
print("ball support force, first 4 worlds:", per_world_ball[:4])

# cross-check world 0 against C MuJoCo's mj_contactForce on the copied MjData
mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d, world_id=0)
print("get_data_into(world 0): mjd.ncon =", mjd.ncon, "(the pool holds", n, "contacts in total)")
c_force = np.zeros(6)
fz_c = 0.0
for i in range(mjd.ncon):
    mujoco.mj_contactForce(mjm, mjd, i, c_force)          # contact frame: normal first
    if mjd.contact.geom[i][1] == ball_geom:
        fz_c += (c_force[:3] @ mjd.contact.frame[i].reshape(3, 3))[2]
print(f"C mj_contactForce, ball: {fz_c:7.3f} N")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 6. Starvation: a shared pool is not fair
Two worlds, six boxes each, a pool of 8 slots.

In [ ]:
# Example 9.5 - ex5_overflow_starvation.py
"""Example 9.5 - a shared pool is not fair: one world uses up the contacts, the other starves.

Six boxes rest on a plane in 2 worlds. With nconmax=4, njmax=8 the pool has naconmax = 8 slots
for 2 x 24 contacts. We print the counters and the decoded overflow bits, then compare box
heights against a run with nconmax=48, njmax=128.
Run: python ex5_overflow_starvation.py [--dump-json PATH]
  --dump-json PATH writes box heights and overflow bits per frame (replay player + LED panel).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

BOXES = "".join(f'<body pos="{0.25 * i} 0 0.09"><freejoint/>'
                f'<geom type="box" size=".1 .1 .1"/></body>'
                for i in range(6))
SIX_BOXES = f'<mujoco><worldbody><geom type="plane" size="5 5 .1"/>{BOXES}</worldbody></mujoco>'

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 2           # the lesson needs exactly two worlds competing for one small pool
NSTEP = 200          # 0.4 s at the default timestep 0.002

mjm = mujoco.MjModel.from_xml_string(SIX_BOXES)
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0      # silence device printf warnings; bits are still recorded (Ex 9.10)


def run(nconmax, njmax, verbose):
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
    print(f"\nnconmax={nconmax}, njmax={njmax} -> naconmax={d.naconmax}, njmax={d.njmax}")
    z, bits, counters = [], [], []
    for i in range(NSTEP):
        mjw.step(m, d)
        if i % 4 == 0:      # decimated host reads: this example is about the counters
            z.append(d.qpos.numpy()[:, 2::7].round(3).tolist())     # z of the 6 free joints
            bits.append(d.overflow.numpy().tolist())
            counters.append([int(d.nacon.numpy()[0]), int(d.ncollision.numpy()[0])]
                            + d.nefc.numpy().tolist())
        if verbose and i < 3:
            names = [mjw.OverflowType(int(b)).name for b in d.overflow.numpy()]
            print(f"step {i}: nacon={d.nacon.numpy()} ncollision={d.ncollision.numpy()}"
                  f" nefc={d.nefc.numpy()} overflow={names}")
    print("box z after 0.4 s:\n", d.qpos.numpy()[:, 2::7])
    print("overflow:", [mjw.OverflowType(int(b)).name for b in d.overflow.numpy()])
    return z, bits, counters


za, bits_a, cnt_a = run(4, 8, verbose=True)
zb, bits_b, cnt_b = run(48, 128, verbose=False)

if args.dump_json:
    dt = 4 * float(mjm.opt.timestep)
    frames = [[a[0], a[1], b[0], b[1]] for a, b in zip(za, zb)]
    out = {"nworld": 4, "dt": dt, "kind": "boxes", "zmin": -1.0, "zmax": 0.3,
           "labels": ["nconmax=4, world 0", "nconmax=4, world 1",
                      "nconmax=48, world 0", "nconmax=48, world 1"],
           "frames": frames, "overflow": bits_a, "counters": cnt_a,
           "counters_names": ["nacon", "ncollision", "nefc w0", "nefc w1"],
           "naconmax": 8, "njmax": 8}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 7. Detecting and decoding overflow
The default `njmax = 64` is too small for a 6-box tower (96 rows). The bits are sticky until `reset_data`.

In [ ]:
# Example 9.6 - ex6_default_overflow.py
"""Example 9.6 - the default budgets overflow on a modest scene: box_stack(6).

A 6-box tower needs 24 contacts x 4 pyramidal rows = 96 efc rows per world, but the default
njmax for a rigid model is 64. The tower sinks, then the solver throws it into the air. With
njmax=128 it simply stands. The bits stay lit after the cause is gone (sticky).
Run: python ex6_default_overflow.py [--dump-json PATH]
  --dump-json PATH writes per-step d.overflow, nefc, nacon and box heights (LED panel).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 300                                  # 1.5 s at dt = 0.005

mjm = mujoco.MjModel.from_xml_string(box_stack(6))
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0                      # bits only, no printf (see Example 9.10)

dump = {}
for label, njmax in (("defaults", None), ("njmax=128", 128)):
    d = mjw.make_data(mjm, nworld=NWORLD, njmax=njmax)     # nconmax left at its default
    print(f"\n{label}: naconmax={d.naconmax} ({d.naconmax // NWORLD} per world), njmax={d.njmax}")
    rec = {"overflow": [], "nefc": [], "nacon": [], "z": []}
    first = None
    for i in range(NSTEP):
        mjw.step(m, d)
        if first is None and d.overflow.numpy()[0]:
            first = i
            nefc0 = int(d.nefc.numpy()[0])
            print(f"  first overflow at step {i}: nefc[w0]={nefc0} > njmax={d.njmax}")
        if i % 5 == 0:
            rec["overflow"].append(d.overflow.numpy()[:4].tolist())
            rec["nefc"].append(d.nefc.numpy()[:4].tolist())
            rec["nacon"].append(int(d.nacon.numpy()[0]))
            rec["z"].append(d.qpos.numpy()[0, 2::7].round(3).tolist())
        if i in (20, 120, 160) and label == "defaults":
            ov = int(d.overflow.numpy()[0])
            nacon, nefc0 = int(d.nacon.numpy()[0]), int(d.nefc.numpy()[0])
            print(f"  step {i:3d}: nacon={nacon:3d} nefc[w0]={nefc0:3d}"
                  f"  overflow[w0]={ov} = {mjw.OverflowType(ov).name}")
    ov = d.overflow.numpy()
    print("  top box z, world 0:", round(float(d.qpos.numpy()[0, 2 + 7 * 5]), 3))
    print("  nefc per world (last step):", d.nefc.numpy()[:4], "...")
    print("  overflow per world:", ov[:4], "->", mjw.OverflowType(int(ov[0])).name)
    print("  NEFC set:", bool(ov[0] & mjw.OverflowType.NEFC))
    dump[label] = rec

if args.dump_json:
    out = {"nworld": 4, "dt": 5 * float(mjm.opt.timestep), "njmax_default": 64,
           "naconmax_default": 48 * NWORLD, "nworld_run": NWORLD,
           "bits": [t.name for t in mjw.OverflowType], "values": [int(t) for t in mjw.OverflowType],
           "runs": dump}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(dump["defaults"]["nefc"]), "frames per run")

Decode the bits by looping over `mjw.OverflowType` (never a hand-written list).

In [ ]:
# Example 9.7 - ex7_decode_overflow.py
"""Example 9.7 - describe_overflow(d): decode d.overflow by looping over mjw.OverflowType.

The catalogue of bits is generated from the enum (never a hand-written list), so new bits in a
later release show up automatically. Three worlds share a tiny pool; world 2 starts with its
boxes in the air. The function is added to the running project, my_env.py.
Run: python ex7_decode_overflow.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit."""
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]   # single-bit members only
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


print("OverflowType catalogue (generated from the enum):")
for bit in mjw.OverflowType:          # Python >= 3.11 iterates single-bit members: no NONE, no ALL
    print(f"  1 << {int(bit).bit_length() - 1:2d} = {int(bit):5d}  {bit.name}")
print(f"  ALL = {int(mjw.OverflowType.ALL)}")

BOXES = "".join(f'<body pos="{0.25 * i} 0 0.09"><freejoint/>'
                f'<geom type="box" size=".1 .1 .1"/></body>'
                for i in range(6))
SIX_BOXES = f'<mujoco><worldbody><geom type="plane" size="5 5 .1"/>{BOXES}</worldbody></mujoco>'

NWORLD = 3
mjm = mujoco.MjModel.from_xml_string(SIX_BOXES)
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0

d = mjw.make_data(mjm, nworld=NWORLD, nconmax=48, njmax=128)
mjw.step(m, d)
print("\nwell sized:", describe_overflow(d))

d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=8)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[2, 2::7] += 1.0                  # world 2: all six boxes start 1 m higher (no contacts)
d.qpos.assign(qpos)
mjw.step(m, d)
print("\nundersized, raw d.overflow:", d.overflow.numpy(), " nefc:", d.nefc.numpy())
print(describe_overflow(d))
mjw.reset_data(m, d)                  # the only thing that clears the sticky bits
print("\nafter reset_data:", describe_overflow(d))

The same function from the running project:

In [ ]:
import importlib
import my_env
importlib.reload(my_env)
print(my_env.describe_overflow(d))

## 8. A sizing recipe
Measure the worst case, add 25%, round up to the ladder, check periodically.

In [ ]:
# Example 9.8 - ex8_size_from_rollout.py
"""Example 9.8 - size nconmax and njmax from a measurement rollout (the sizing recipe).

1. Run the scene with generous budgets. 2. Track the worst case per world: contacts (bincount of
contact.worldid) and constraint rows (nefc). 3. Add a 25% margin and round up to the valid-size
ladder. 4. Re-run with the new sizes and confirm there is no overflow.
Run: python ex8_size_from_rollout.py [--dump-json PATH]
  --dump-json PATH writes per-step contact and row counts of 8 worlds (sizing chart).
"""
import argparse
import json
import math

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


# the same ladder io.py uses for its defaults: 16, 24, 32, 48, 64, 96, ... 8192
LADDER = [(2 + k % 2) * 2 ** (k // 2 + 3) for k in range(19)]


def round_up_ladder(x):
    return next((v for v in LADDER if v >= x), int(x))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 300

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0


def rollout(nconmax, njmax, seed=0):
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
    rng = np.random.default_rng(seed)
    qpos = np.tile(mjm.qpos0, (NWORLD, 1))
    for i in range(5):                                    # shift each box sideways a little
        qpos[:, 7 * i:7 * i + 2] += rng.uniform(-0.09, 0.09, (NWORLD, 2)) * (i > 0)
    d.qpos.assign(qpos)
    max_con = np.zeros(NWORLD, dtype=int)
    max_efc = np.zeros(NWORLD, dtype=int)
    trace = {"con": [], "efc": []}
    for i in range(NSTEP):
        mjw.step(m, d)
        n = min(int(d.nacon.numpy()[0]), d.naconmax)      # one host read per step: measuring only
        con = np.bincount(d.contact.worldid.numpy()[:n], minlength=NWORLD)
        max_con = np.maximum(max_con, con)
        max_efc = np.maximum(max_efc, d.nefc.numpy())
        if i % 5 == 0:
            trace["con"].append(con[:8].tolist())
            trace["efc"].append(d.nefc.numpy()[:8].tolist())
    return d, max_con, max_efc, trace


d, max_con, max_efc, trace = rollout(nconmax=256, njmax=1024)
print(f"measurement run: nworld={NWORLD}, {NSTEP} steps, nconmax=256, njmax=1024")
print("worst contacts per world:", max_con[:8], " max:", max_con.max())
print("worst nefc per world:    ", max_efc[:8], " max:", max_efc.max())
print("measurement run overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

nconmax = round_up_ladder(math.ceil(1.25 * max_con.max()))
njmax = round_up_ladder(math.ceil(1.25 * max_efc.max()))
print(f"recommended: nconmax={nconmax}, njmax={njmax}  (max * 1.25, rounded up to the ladder)")
print("mjw defaults for this model would be: nconmax=48, njmax=64")

d2, _, _, _ = rollout(nconmax, njmax, seed=1)               # check on a fresh draw of the scene
print("check run with the new sizes, other seed: overflow",
      "none" if not d2.overflow.numpy().any() else d2.overflow.numpy())

mjd = mujoco.MjData(mjm)                                  # C MuJoCo reference, unperturbed tower
worst = [0, 0]
for _ in range(NSTEP):
    mujoco.mj_step(mjm, mjd)
    worst = [max(worst[0], mjd.ncon), max(worst[1], mjd.nefc)]
print(f"C MuJoCo, unperturbed tower: max ncon={worst[0]}, max nefc={worst[1]}")

if args.dump_json:
    out = {"dt": 5 * float(mjm.opt.timestep), "nworld_run": NWORLD, "con": trace["con"],
           "efc": trace["efc"], "max_con": int(max_con.max()), "max_efc": int(max_efc.max()),
           "nconmax": nconmax, "njmax": njmax, "default_nconmax": 48, "default_njmax": 64}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(trace["con"]), "frames")

## 9. What oversizing costs
Memory formulas checked against the real allocation.

In [ ]:
# Example 9.9 - ex9_memory_estimate.py
"""Example 9.9 - estimate device memory from nworld, nconmax, njmax and nv, then check it.

Formulas: dense efc.J is nworld * njmax_pad * nv_pad floats; the contact pool costs a fixed
number of bytes per slot times naconmax; the Newton Hessian h (solver workspace, not in Data)
is nworld * nv_pad^2 floats (plus an equal-size hfactor when nv > 48). We compare with the summed
bytes of every array in a real Data at small nworld, then extrapolate (memory is linear in nworld) to 1k, 8k and 32k worlds.
Run: python ex9_memory_estimate.py
"""
import dataclasses

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def nbytes_tree(obj, prefix=""):
    """{name: bytes} for every wp.array inside a (nested) dataclass such as mjw.Data."""
    out = {}
    for f in dataclasses.fields(obj):
        v = getattr(obj, f.name)
        if isinstance(v, wp.array):
            out[prefix + f.name] = v.size * wp.types.type_size_in_bytes(v.dtype)  # no .nbytes
        elif dataclasses.is_dataclass(v):
            out.update(nbytes_tree(v, prefix + f.name + "."))
    return out


def round_up(x, k):
    return (x + k - 1) // k * k


def estimate(nworld, nconmax, njmax, nv, nmaxpyramid=4, nflex=0):
    njmax_pad = round_up(njmax, 16)
    nv_pad = round_up(nv, 4) if nv <= 48 else round_up(nv + 1, 16)   # +1: Newton's augmented row
    # dist pos frame includemargin friction solref solreffriction solimp adhesion surfacevel
    floats = 1 + 3 + 9 + 1 + 5 + 2 + 2 + 5 + 1 + 3
    # dim geom efc_address worldid type geomcollisionid
    ints = 1 + 2 + nmaxpyramid + 1 + 1 + 1
    ints += 6 if nflex else 0                          # flex/elem/vert pairs only exist with flex
    per_slot = 4 * (floats + ints)
    return {"contact": nworld * nconmax * per_slot,
            "efc.J": nworld * njmax_pad * nv_pad * 4,
            "h": nworld * nv_pad * nv_pad * 4,
            "per_slot": per_slot, "njmax_pad": njmax_pad, "nv_pad": nv_pad}


mjm = mujoco.MjModel.from_xml_string(box_stack(5))
NCONMAX, NJMAX = 32, 128           # the sizes Example 9.8 recommends for box_stack(5)
print(f"model: box_stack(5), nv={mjm.nv} (dense J: nv <= 32), nconmax={NCONMAX}, njmax={NJMAX}")

totals = {}
for nw in (8, 16):
    d = mjw.make_data(mjm, nworld=nw, nconmax=NCONMAX, njmax=NJMAX)
    sizes = nbytes_tree(d)
    totals[nw] = sum(sizes.values())
    if nw == 8:
        est = estimate(nw, NCONMAX, NJMAX, mjm.nv)
        contact = sum(v for k, v in sizes.items() if k.startswith("contact."))
        efc = sum(v for k, v in sizes.items() if k.startswith("efc."))
        print(f"nworld=8: efc.J {d.efc.J.shape} -> measured {sizes['efc.J']:,} B,"
              f" estimate {est['efc.J']:,} B")
        print(f"          contact pool {d.naconmax} slots x {est['per_slot']} B -> measured"
              f" {contact:,} B, estimate {est['contact']:,} B")
        print(f"          all efc.* {efc:,} B | whole Data {totals[nw]:,} B"
              f" | {len(sizes)} arrays")
        top = sorted(sizes.items(), key=lambda kv: -kv[1])[:4]
        print("          largest arrays:", ", ".join(f"{k} {v:,}" for k, v in top))

per_world = (totals[16] - totals[8]) / 8          # memory is linear in nworld
fixed = totals[8] - 8 * per_world
print(f"Data bytes = {fixed:,.0f} + {per_world:,.0f} * nworld")

print("\n nworld   contacts MB   efc.J MB   Newton h MB   whole Data MB (extrapolated)")
for nw in (1024, 8192, 32768):
    e = estimate(nw, NCONMAX, NJMAX, mjm.nv)
    total = fixed + per_world * nw
    print(f"{nw:7d}   {e['contact'] / 2**20:11.1f}   {e['efc.J'] / 2**20:8.1f}"
          f"   {e['h'] / 2**20:11.1f}   {total / 2**20:13.1f}")

big = estimate(8192, 48, 512, 60)
print(f"\nwhat-if: nworld=8192, njmax=512, nv=60 -> dense J would be {big['efc.J'] / 2**30:.2f} GiB"
      f" (nv_pad={big['nv_pad']}); MJWarp uses sparse J for nv > 32 (jacobian=AUTO)")

### The sizing calculator
The page's calculator widget as a function, with optional sliders.

In [ ]:
# The sizing calculator of Figure 13 as a Python function (formulas from Example 9.9).
def sizing_calculator(nworld, nconmax, njmax, nv, condim=3, cone="pyramidal", solver="newton"):
    """Print the derived sizes and the memory of the contact pool, efc.J and Newton's h (+ hfactor when nv > 48).

    Exact for a dense J (nv <= 32) and nv <= 48; for nv > 32 the sparse J figure is an upper bound
    (make_data's default njmax_nnz is usually much smaller than njmax * nv).
    """
    def round_up(x, k):
        return (x + k - 1) // k * k
    newton = solver == "newton"
    naconmax = nconmax * nworld
    njmax_pad = round_up(njmax, 16)
    nv_pad = round_up(nv, 4) if nv <= 48 else round_up(nv + int(newton), 16)
    nmaxpyramid = max(1, 2 * (condim - 1))
    rows = 1 if condim == 1 else (2 * (condim - 1) if cone == "pyramidal" else condim)
    slot = 4 * (32 + 6 + nmaxpyramid)              # bytes per pool slot without flex
    contacts = naconmax * slot
    sparse = nv > 32                                # jacobian=AUTO rule (io.is_sparse)
    J = 2 * 4 * nworld * njmax * nv if sparse else nworld * njmax_pad * nv_pad * 4
    h = (2 if nv > 48 else 1) * nworld * nv_pad * nv_pad * 4 if newton else 0   # + hfactor when nv > 48
    print(f"naconmax = {naconmax:,}   njmax_pad = {njmax_pad}   nv_pad = {nv_pad}")
    print(f"rows per contact = {rows} -> {njmax // max(rows, 1)} contacts fit in njmax = {njmax}")
    print(f"contact pool     {contacts / 2**20:10.1f} MiB  ({slot} B per slot)")
    print(f"efc.J {'sparse<=' if sparse else 'dense   '}   {J / 2**20:10.1f} MiB")
    print(f"Newton h{' + hfactor' if nv > 48 else '         '}{h / 2**20:10.1f} MiB  (solver workspace, not in Data)")

sizing_calculator(nworld=8192, nconmax=32, njmax=128, nv=30)

# Optional: sliders (ipywidgets is preinstalled on Colab)
try:
    import ipywidgets as widgets
    widgets.interact(sizing_calculator,
                     nworld=widgets.IntSlider(8192, 1024, 65536, 1024),
                     nconmax=widgets.IntSlider(32, 4, 512, 4), njmax=widgets.IntSlider(128, 16, 2048, 16),
                     nv=widgets.IntSlider(30, 1, 120), condim=[1, 3, 4, 6],
                     cone=["pyramidal", "elliptic"], solver=["newton", "cg"])
except ImportError:
    print("skipped: ipywidgets not installed")

## 10. Silencing warnings without losing the bits
`m.opt.warn_overflow` gates only the device printf; it is read through `wp.static`, so each new value recompiles some kernels. In a notebook the device text may show up late or only in the runtime log.

In [ ]:
# Example 9.10 - ex10_warn_mask.py
"""Example 9.10 - silence one warning with m.opt.warn_overflow, and see that the bit is still set.

The overflow warnings are wp.printf calls inside kernels: C-level output, buffered apart from
Python's print. flush_device_prints() pushes it out so the order on screen is the real order.
warn_overflow is read through wp.static, so each new value compiles new kernel variants.
Run: python ex10_warn_mask.py
"""
import ctypes
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def flush_device_prints():
    """Wait for the device, then flush C stdio so wp.printf text appears before our next line."""
    wp.synchronize()
    sys.stdout.flush()
    try:
        ctypes.CDLL(None).fflush(None)         # Linux/macOS; harmless to skip elsewhere
    except (OSError, AttributeError):
        pass


BOXES = "".join(f'<body pos="{0.25 * i} 0 0.09"><freejoint/>'
                f'<geom type="box" size=".1 .1 .1"/></body>'
                for i in range(6))
SIX_BOXES = f'<mujoco><worldbody><geom type="plane" size="5 5 .1"/>{BOXES}</worldbody></mujoco>'

mjm = mujoco.MjModel.from_xml_string(SIX_BOXES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=8)
print("default m.opt.warn_overflow =", m.opt.warn_overflow, "=", repr(mjw.OverflowType.ALL))

print("\n--- step 1: all warnings on")
flush_device_prints()
mjw.step(m, d)
flush_device_prints()

m.opt.warn_overflow &= ~mjw.OverflowType.NEFC      # new static value -> new kernel variants
print("\n--- step 2: NEFC silenced, warn_overflow =", int(m.opt.warn_overflow))
flush_device_prints()
mjw.step(m, d)
flush_device_prints()
print("bits per world:", [mjw.OverflowType(int(b)).name for b in d.overflow.numpy()])
print("NEFC still recorded in world 0:", bool(d.overflow.numpy()[0] & mjw.OverflowType.NEFC))

m.opt.warn_overflow = 0                            # silence everything (bool False works too)
print("\n--- step 3: warn_overflow = 0, nothing printed")
flush_device_prints()
mjw.step(m, d)
flush_device_prints()
print("raw d.overflow:", d.overflow.numpy(), "(sticky until reset_data)")

## 11. Not every contact is a constraint: ContactType bits
Collision sensors add SENSOR contacts: they use pool slots but no constraint rows.

In [ ]:
# Example 9.11 - ex11_sensor_contacts.py
"""Example 9.11 - collision sensors put SENSOR contacts in the pool, but no constraint rows.

Two balls rest on a floor 0.3 m apart. A <distance> sensor between the balls makes the
narrowphase write a contact with only the SENSOR bit (dist = +0.3). A second sensor between the
floor and ball_a marks a real contact CONSTRAINT|SENSOR. nefc counts only CONSTRAINT contacts.
Run: python ex11_sensor_contacts.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import ContactType   # private API: not exported by mujoco_warp

np.set_printoptions(precision=4, suppress=True)

SENSOR_SCENE = """
<mujoco model="sensor_contacts">
  <option timestep="0.002"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="a" pos="0 0 0.1"><freejoint/><geom name="ball_a" type="sphere" size="0.1"/></body>
    <body name="b" pos="0.5 0 0.1"><freejoint/><geom name="ball_b" type="sphere" size="0.1"/></body>
  </worldbody>
  <sensor>
    <distance name="gap_ab" geom1="ball_a" geom2="ball_b" cutoff="1"/>
    <distance name="gap_floor_a" geom1="floor" geom2="ball_a" cutoff="1"/>
  </sensor>
</mujoco>
"""

NWORLD = 2
mjm = mujoco.MjModel.from_xml_string(SENSOR_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
for _ in range(10):
    mjw.step(m, d)

names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
n = min(int(d.nacon.numpy()[0]), d.naconmax)
c = d.contact
print(f"nacon = {n}, nefc per world = {d.nefc.numpy()}")
print(" cid  w  pair             dist     type                efc_address")
for i in range(n):
    g0, g1 = c.geom.numpy()[i]
    t = ContactType(int(c.type.numpy()[i]))
    pair = names[g0] + "-" + names[g1]
    print(f"{i:4d} {c.worldid.numpy()[i]:2d}  {pair:15s} {c.dist.numpy()[i]:7.4f}"
          f"   {t.name:18s}  {c.efc_address.numpy()[i]}")

is_sensor = (c.type.numpy()[:n] & ContactType.SENSOR) != 0
is_constraint = (c.type.numpy()[:n] & ContactType.CONSTRAINT) != 0
w0 = c.worldid.numpy()[:n] == 0
print("world 0: SENSOR contacts", int((is_sensor & w0).sum()),
      "| CONSTRAINT contacts", int((is_constraint & w0).sum()),
      "| 4 rows each ->", 4 * int((is_constraint & w0).sum()),
      "== nefc[0] =", int(d.nefc.numpy()[0]))
print("sensordata (gap_ab, gap_floor_a) per world:\n", d.sensordata.numpy())

mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d, world_id=0)          # exports CONSTRAINT | PASSIVE contacts only
print("get_data_into(world 0): mjd.ncon =", mjd.ncon, "(the SENSOR-only contact is not exported)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Exercise 9.1: size a 10-box tower for 8192 worlds
Measure `box_stack(10)` with 4 worlds for 200 steps, pick `nconmax`/`njmax` (worst x 1.25, rounded up to the ladder), confirm with a check run.

In [ ]:
# Exercise 9.1: size box_stack(10) for 8192 worlds (box_stack is defined by Example 9.8 above)
import math
LADDER = [(2 + k % 2) * 2 ** (k // 2 + 3) for k in range(19)]
TARGET_NWORLD = 8192
mjm = mujoco.MjModel.from_xml_string(box_stack(10))
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0
worst_con = worst_efc = None             # measure with 4 worlds, 200 steps, nconmax=256, njmax=1024
nconmax = njmax = None                   # 1.25 x worst, rounded up to the ladder
if worst_con is not None and nconmax is not None:
    naconmax = nconmax * TARGET_NWORLD
    assert nconmax in LADDER and njmax in LADDER
    assert nconmax >= 1.25 * worst_con and njmax >= 1.25 * worst_efc
    assert naconmax == nconmax * TARGET_NWORLD
    # ... and a 4-world check run with (nconmax, njmax): assert not d.overflow.numpy().any()
    print("all checks passed")
else:
    print("skipped: fill in worst_con, worst_efc, nconmax and njmax, then run this cell again")

In [ ]:
#@title Exercise 9.1: solution (double-click to show the code)
import math
LADDER = [(2 + k % 2) * 2 ** (k // 2 + 3) for k in range(19)]
TARGET_NWORLD = 8192
mjm = mujoco.MjModel.from_xml_string(box_stack(10))
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0


def measure(nconmax, njmax, nworld=4, nstep=200):
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax)
    worst_con, worst_efc = 0, 0
    for _ in range(nstep):
        mjw.step(m, d)
        n = min(int(d.nacon.numpy()[0]), d.naconmax)
        per_world = np.bincount(d.contact.worldid.numpy()[:n], minlength=nworld)
        worst_con = max(worst_con, int(per_world.max()))
        worst_efc = max(worst_efc, int(d.nefc.numpy().max()))
    return worst_con, worst_efc, d


worst_con, worst_efc, _ = measure(nconmax=256, njmax=1024)
nconmax = next(v for v in LADDER if v >= math.ceil(1.25 * worst_con))
njmax = next(v for v in LADDER if v >= math.ceil(1.25 * worst_efc))
naconmax = nconmax * TARGET_NWORLD
print(f"worst: {worst_con} contacts, {worst_efc} rows -> nconmax={nconmax}, njmax={njmax}, naconmax={naconmax}")
assert nconmax in LADDER and njmax in LADDER
assert nconmax >= 1.25 * worst_con and njmax >= 1.25 * worst_efc
assert naconmax == nconmax * TARGET_NWORLD
_, _, d = measure(nconmax, njmax)
assert not d.overflow.numpy().any(), "check run overflowed"
print("all checks passed")

## Exercise 9.2: count contacts per world on the device
One thread per pool slot, an atomic add per world, compared with `np.bincount`.

In [ ]:
# Exercise 9.2: count contacts per world on the device (CONTACT_SCENE is defined by Example 9.1)
@wp.kernel
def count_contacts(nacon: wp.array(dtype=int), naconmax: int,
                   worldid: wp.array(dtype=int), counts: wp.array(dtype=int)):
    cid = wp.tid()
    # your code: skip unused slots, then one atomic add (replace the return below)
    return


IMPLEMENTED = False                           # set to True once the kernel body is written
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(CONTACT_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[1::2, 9] += 0.5                          # odd worlds: lift the box
d.qpos.assign(qpos)
mjw.forward(m, d)
if IMPLEMENTED:
    counts = wp.zeros(d.nworld, dtype=int)
    wp.launch(count_contacts, dim=d.naconmax,
              inputs=[d.nacon, d.naconmax, d.contact.worldid], outputs=[counts])
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    assert np.array_equal(counts.numpy(), np.bincount(d.contact.worldid.numpy()[:n], minlength=d.nworld))
    print("match")
else:
    print("skipped: write the kernel body and set IMPLEMENTED = True")

In [ ]:
#@title Exercise 9.2: solution (double-click to show the code)
@wp.kernel
def count_contacts(nacon: wp.array(dtype=int), naconmax: int,
                   worldid: wp.array(dtype=int), counts: wp.array(dtype=int)):
    cid = wp.tid()                                   # one thread per pool slot
    if cid >= wp.min(nacon[0], naconmax):            # slot unused (or counter past capacity)
        return
    wp.atomic_add(counts, worldid[cid], 1)


NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(CONTACT_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[1::2, 9] += 0.5
d.qpos.assign(qpos)
mjw.forward(m, d)
counts = wp.zeros(d.nworld, dtype=int)
wp.launch(count_contacts, dim=d.naconmax,
          inputs=[d.nacon, d.naconmax, d.contact.worldid], outputs=[counts])
n = min(int(d.nacon.numpy()[0]), d.naconmax)
expected = np.bincount(d.contact.worldid.numpy()[:n], minlength=d.nworld)
print("kernel:  ", counts.numpy())
print("bincount:", expected)
assert np.array_equal(counts.numpy(), expected)
print("match")

## Stretch: the smallest nconmax for TERRAIN
Compare the heightfield defaults with what the scene needs. Reference result on CPU (2 worlds, 300 steps): defaults 192/1536; no overflow from nconmax = 6 (at most 6 contacts and 24 rows per world). With a 25% margin and the ladder (which starts at 16): nconmax = 16 (1.25 x 6 = 7.5), njmax = 32 (1.25 x 24 = 30).

In [ ]:
# Stretch: the smallest nconmax with no overflow for TERRAIN (plan model M7).
# The first run compiles the heightfield and convex colliders (about a minute on CPU).
from mujoco_warp._src import io as mjw_io     # private API: only to print the defaults
TERRAIN = """
<mujoco model="terrain">
  <option timestep="0.005"/>
  <asset>
    <hfield name="bumps" nrow="5" ncol="5" size="1 1 .2 .1"
      elevation="0 0 0 0 0  0 .5 1 .5 0  0 1 0 1 0  0 .5 1 .5 0  0 0 0 0 0"/>
    <mesh name="tetra" vertex="0 0 0  .2 0 0  0 .2 0  0 0 .2"/>
  </asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="ground" type="hfield" hfield="bumps"/>
    <body name="ball" pos="0.2 0.1 .6"><freejoint/><geom type="sphere" size=".08"/></body>
    <body name="ellipsoid" pos="-0.3 0.2 .6"><freejoint/><geom type="ellipsoid" size=".1 .07 .05"/></body>
    <body name="cyl" pos="0.3 -0.3 .6"><freejoint/><geom type="cylinder" size=".06 .05"/></body>
    <body name="tetra" pos="-0.3 -0.3 .6"><freejoint/><geom type="mesh" mesh="tetra"/></body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(TERRAIN)
print("defaults:", mjw_io._default_nconmax(mjm), mjw_io._default_njmax(mjm))
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0
# your sweep here: for nconmax in [...]: make_data(nworld=2, nconmax=..., njmax=128), 300 steps,
# record max d.nacon and print describe_overflow(d)